#### 06. Tier 2 features

Tier 2 describes the rain at the neighbouring stations up to and including the prediction time. Rain usually reaches a station from somewhere nearby, so a storm already falling a few kilometres away is the main warning sign when the station itself is still dry, the case where the baselines and Tier 1 do worst.

For every station and 5-minute slot the features summarise the last 15 and 30 minutes of rain at other stations: a distance-weighted mean (weights 1/d², within 10 km) and the wettest station within 10 km, a ring 10 to 25 km out, the share of neighbours within 5 and 10 km that are wet, the distance to the nearest station that rained in the last 15 minutes, the mean rain in each compass quadrant within 25 km (storms in Singapore mostly move from west to east, Task 1), and the island-wide share of wet stations and mean rain. Averages only count neighbours that have a reading, and `t2_nbrs_10km` says how many there were.

The computation puts all stations of one slot side by side in one row, so each feature becomes a small matrix product applied with `mapInPandas`. The result is saved to `task2/data/feat_t2/` and joined into the train, validation and test tables by notebook 03.

**Setup**

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "../code")
import pandas as pd
from pyspark.sql import functions as F
from pyspark.ml.evaluation import BinaryClassificationEvaluator

import config as C
import features as FT

spark = C.get_spark("task2-tier2")

**Building the features**

In [2]:
feat_t1 = spark.read.parquet(str(C.FEAT_T1))
FT.add_tier2(spark, feat_t1).write.mode("overwrite").partitionBy("year").parquet(str(C.FEAT_T2))

feat_t2 = spark.read.parquet(str(C.FEAT_T2))
T2 = [c for c in feat_t2.columns if c.startswith("t2_")]
print(f"{feat_t2.count():,} rows, {len(T2)} features:", T2)

48,409,778 rows, 15 features: ['t2_idw_rain_15', 't2_idw_rain_30', 't2_idw_trend', 't2_far_rain_30', 't2_max_rain_30', 't2_wet_frac_5km', 't2_wet_frac_10km', 't2_dist_rain_km', 't2_rain_30_n', 't2_rain_30_e', 't2_rain_30_s', 't2_rain_30_w', 't2_island_wet_frac', 't2_island_rain_30', 't2_nbrs_10km']


**A storm reaching S07**

S07 on 14 April 2023, every 15 minutes. Its own rain starts at 16:15, but a station 3.2 km away is already wet at 15:50, with rain to the west; afterwards the rain moves to the east quadrant.

In [3]:
cols = ["t2_idw_rain_30", "t2_max_rain_30", "t2_wet_frac_10km", "t2_dist_rain_km", "t2_rain_30_w", "t2_rain_30_e", "t2_island_wet_frac"]
(feat_t2.filter((F.col("station_id") == "S07") & F.col("slot").between(660850, 660880))
        .join(feat_t1.select("station_id", "slot", "t1_rain_30"), ["station_id", "slot"])
        .withColumn("time", F.date_format(F.timestamp_seconds(F.lit(C.ORIGIN_EPOCH) + F.col("slot") * C.SLOT_SECONDS), "HH:mm"))
        .orderBy("slot").select("time", "t1_rain_30", *cols).toPandas().iloc[::3].set_index("time").round(3))

,t1_rain_30,t2_idw_rain_30,t2_max_rain_30,t2_wet_frac_10km,t2_dist_rain_km,t2_rain_30_w,t2_rain_30_e,t2_island_wet_frac
time,,,,,,,,
14:50,0.0,0.000,0.0,0.000,30.000,0.000,0.000,0.000
15:05,0.0,0.000,0.0,0.000,30.000,0.000,0.000,0.000
15:20,0.0,0.000,0.0,0.000,30.000,0.000,0.000,0.000
15:35,0.0,0.000,0.0,0.000,30.000,0.000,0.000,0.000
15:50,0.0,0.035,2.2,0.077,3.239,0.122,0.000,0.058
16:05,0.0,0.794,18.2,0.077,3.239,0.904,0.000,0.058
16:20,2.8,1.779,19.4,0.231,2.625,0.991,0.104,0.145
16:35,3.2,3.141,9.4,0.436,1.652,0.217,0.791,0.261
16:50,0.4,2.525,16.2,0.513,2.625,0.009,1.591,0.319


**Which features carry signal**

As for Tier 1, each feature on its own ranks a 20 % sample of the training rows by the chance of rain in the next 30 minutes (AUC-PR), on all rows and on the rows where the station is dry now. A feature that cannot rank scores the event rate.

In [4]:
sample = (spark.read.parquet(str(C.LABELS))
               .filter(F.col("year").between(*C.TRAIN_YEARS))
               .sample(fraction=0.2, seed=C.SEED)
               .join(feat_t2, ["station_id", "slot", "year"])
               .withColumn("wet", (F.col("label") > 0).cast("double"))
               .select("wet", "regime", *T2)
               .cache())
subsets = {"all": sample, "dry now": sample.filter(F.col("regime") == "dry_now")}

means = sample.groupBy("wet").agg(*[F.avg(c).alias(c) for c in T2]).toPandas().set_index("wet")
direction = {c: 1 if means.loc[1.0, c] >= means.loc[0.0, c] else -1 for c in T2}
ev = BinaryClassificationEvaluator(labelCol="wet", metricName="areaUnderPR")

rows = []
for c in T2:
    row = {"feature": c, "direction": "+" if direction[c] > 0 else "−"}
    for name, df in subsets.items():
        row[f"AUC-PR {name}"] = ev.setRawPredictionCol("s").evaluate(df.withColumn("s", F.col(c) * direction[c]))
    rows.append(row)
rates = {f"AUC-PR {name}": df.agg(F.avg("wet")).first()[0] for name, df in subsets.items()}
signal = pd.DataFrame(rows + [{"feature": "event rate (no skill)", "direction": "", **rates}]).set_index("feature")
signal.round(4)

,direction,AUC-PR all,AUC-PR dry now
feature,,,
t2_idw_rain_15,+,0.7024,0.3188
t2_idw_rain_30,+,0.6734,0.2782
t2_idw_trend,+,0.4167,0.2784
t2_far_rain_30,+,0.5296,0.2203
t2_max_rain_30,+,0.5665,0.2477
t2_wet_frac_5km,+,0.6067,0.2333
t2_wet_frac_10km,+,0.6745,0.2851
t2_dist_rain_km,−,0.5918,0.2549
t2_rain_30_n,+,0.4760,0.1628


Tier 2 is what the onsets were missing. When the station is dry now, the best Tier 1 feature ranked rain at AUC-PR 0.11; the distance-weighted neighbour rain of the last 15 minutes reaches 0.32, thirteen times the event rate, and every neighbour feature except the station count beats the best Tier 1 feature. Over all rows the same feature is also the strongest single predictor (0.70, against 0.67 for the best Tier 1 feature).

The four compass quadrants carry similar signal on their own (0.16 to 0.20 when dry now). Which direction matters depends on the season, so a directional effect can only show up in the models, not in this one-feature view.